In [ ]:
import requests
import pandas as pd
from sklearn.linear_model import LinearRegression
import joblib
import os

BASE_URL = "http://localhost:3000"

MACHINE_IDS = ["CNC_01", "CNC_02", "PUMP_03", "CONVEYOR_04"]

MODEL_DIR = "models"
os.makedirs(MODEL_DIR, exist_ok=True)


def train_model(machine_id):
    print(f"\nTraining {machine_id}...")

    # Fetch history
    data = requests.get(f"{BASE_URL}/history/{machine_id}").json()
    df = pd.DataFrame(data["readings"])

    # 🔥 IMPORTANT: Only normal data
    df = df[df["status"] == "running"]
    df = df.dropna()

    # Features
    X = df[["rpm", "temperature_C", "vibration_mm_s"]]
    y = df["current_A"]

    # Train
    model = LinearRegression()
    model.fit(X, y)

    # Save
    joblib.dump(model, f"{MODEL_DIR}/model_{machine_id}.pkl")

    print(f"✅ Saved model_{machine_id}.pkl")


if __name__ == "__main__":
    for m in MACHINE_IDS:
        train_model(m)

    print("\n🎉 Training complete!")